Section A: SQL and DataFrames

In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),

(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

Section A: SQL and DataFrames

In [0]:
display(spark.table("parking_sessions_source"))
display(spark.table("parking_sessions_source").select("session_id", "vehicle_type", "parking_fee"))

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,240,Active
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
107,Z03,SUV,2,120,Cancelled
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active


session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
SELECT * FROM parking_sessions_source
WHERE status = 'Completed' AND parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
SELECT vehicle_type,
       SUM(parking_fee) AS total_fees,
       COUNT(*) AS completed_sessions
FROM parking_sessions_source
WHERE status = 'Completed'
GROUP BY vehicle_type;

vehicle_type,total_fees,completed_sessions
Car,1750,8
Bike,160,4
SUV,540,2


In [0]:
from pyspark.sql import functions as F

src_df = spark.read.table("parking_sessions_source")
fee_df = src_df.withColumn("fee_category", F.when(F.col("parking_fee") >= 200, "High").otherwise("Low"))
display(fee_df)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


Section B: Delta Tables and CRUD

In [0]:
%sql
CREATE OR REPLACE TABLE parking_summary
USING DELTA
AS SELECT * FROM parking_sessions_source
WHERE status = 'Completed';

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT INTO parking_sessions_work
VALUES (123, 'Z01', 'Car', 3, 150, 'Active');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE parking_sessions_work
SET status = 'Completed', parking_fee = 280
WHERE session_id = 104;

num_affected_rows
1


In [0]:
%sql
DELETE FROM parking_sessions_work
WHERE status = 'Cancelled';

num_affected_rows
2


In [0]:
%sql
SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,4,280,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


Section C: MERGE and Delta Operations

In [0]:
%sql
SELECT i.*,
       CASE WHEN t.session_id IS NULL THEN 'New' ELSE 'Already exists' END AS record_type
FROM incoming_parking_sessions i
LEFT JOIN parking_sessions_work t ON i.session_id = t.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,record_type
102,Z02,Bike,3,60,Completed,Already exists
104,Z01,SUV,5,300,Completed,Already exists
121,Z03,Car,4,200,Completed,New
122,Z05,Bike,2,40,Active,New


In [0]:
%sql
MERGE INTO parking_sessions_work AS t
USING incoming_parking_sessions AS s
ON t.session_id = s.session_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T04:19:30.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1797602060969920),da3a53c5-9c05-49b4-bc7d-25aeaf8583c2,1106-041422-9v65qjpj-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 13557, p25FileSize -> 3135, numDeletionVectorsRemoved -> 1, minFileSize -> 3135, numAddedFiles -> 1, maxFileSize -> 3135, p75FileSize -> 3135, p50FileSize -> 3135, numAddedBytes -> 3135)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T04:19:29.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(session_id#13381 = session_id#13363)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1797602060969920),da3a53c5-9c05-49b4-bc7d-25aeaf8583c2,1106-041422-9v65qjpj-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 10426, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 4270, materializeSourceTimeMs -> 353, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1371, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2468)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:18:53.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1797602060969920),a3381de4-2a35-4398-be17-aec13fcc5cda,1106-041422-9v65qjpj-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6852, p25FileSize -> 3131, numDeletionVectorsRemoved -> 1, minFileSize -> 3131, numAddedFiles -> 1, maxFileSize -> 3131, p75FileSize -> 3131, p50FileSize -> 3131, numAddedBytes -> 3131)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:18:50.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(status#12675 = Cancelled)""])",null,List(1797602060969920),a3381de4-2a35-4398-be17-aec13fcc5cda,1106-041422-9v65qjpj-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1295, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 827, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 467)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:18:41.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(session_id#12090 = 104)""])",null,List(1797602060969920),9cc10e5d-239d-4e90-808f-ba80b00551cb,1106-041422-9v65qjpj-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4130, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2079, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2660, rewriteTimeMs -> 2023)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:18:31.000Z,143925524137217,azuser7412_mml.local@karthikirisoutlook.on

In [0]:
%sql
SELECT 'before_merge' AS stage, *
FROM parking_sessions_work VERSION AS OF 3
WHERE session_id = 102
UNION ALL
SELECT 'after_merge' AS stage, *
FROM parking_sessions_work
WHERE session_id = 102;

stage,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
before_merge,102,Z02,Bike,2,40,Completed
after_merge,102,Z02,Bike,3,60,Completed


Section D: Tables, Views and Optimization

In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT * FROM parking_sessions_work
WHERE status = 'Completed';

In [0]:
%sql
SELECT * FROM completed_parking_view;


session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


In [0]:
%sql
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT * FROM parking_sessions_work
WHERE status = 'Completed';

In [0]:
spark.sql("CREATE OR REPLACE TEMP VIEW zone_z01_view AS SELECT * FROM parking_sessions_work WHERE zone_id = 'Z01'")
spark.sql("CREATE OR REPLACE TEMP VIEW high_fee_view AS SELECT * FROM parking_sessions_work WHERE parking_fee > 200")
display(spark.sql("SELECT * FROM zone_z01_view"))


session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
114,Z01,SUV,3,180,Completed
123,Z01,Car,3,150,Active
104,Z01,SUV,5,300,Completed


In [0]:
%sql
SELECT z.zone_name, z.city, w.vehicle_type, w.parking_fee
FROM parking_sessions_work w
JOIN parking_zones z ON w.zone_id = z.zone_id
ORDER BY z.zone_name;

zone_name,city,vehicle_type,parking_fee
Airport Parking,Delhi,Car,250
Airport Parking,Delhi,Bike,40
Airport Parking,Delhi,SUV,300
Airport Parking,Delhi,Bike,20
Central Plaza,Hyderabad,Bike,80
Central Plaza,Hyderabad,SUV,180
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,Car,150
Central Plaza,Hyderabad,SUV,300
City Mall,Mumbai,Car,350


In [0]:
detail_before = spark.sql("DESCRIBE DETAIL parking_sessions_work")
display(detail_before)
files_before = detail_before.select("numFiles").collect()[0][0]

display(spark.sql("OPTIMIZE parking_sessions_work"))

detail_after = spark.sql("DESCRIBE DETAIL parking_sessions_work")
display(detail_after)
files_after = detail_after.select("numFiles").collect()[0][0]

print("Files before:", files_before)
print("Files after:", files_after)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,04ff13f2-de75-4e1e-9dae-41ac4ee34f6d,rithika_databricks_7405609940652409.default.parking_sessions_work,null,abfss://unity-catalog-storage@dbstorage6vxx4jf6d74oc.dfs.core.windows.net/7405609940652409/__unitystorage/catalogs/61b034ef-f999-4467-a423-10d5d7c8261f/tables/b2a4dc55-c6b2-4f25-babd-bee6e67b4d16,2026-11-06T04:15:20.598Z,2026-11-06T04:19:30.000Z,List(),List(),1,3135,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b2a4dc55-c6b2-4f25-babd-bee6e67b4d16, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-56591cb8-3714-4e47-8580-f435ffe9518e, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-7261b99b-0e82-4494-bbf8-b72648dbd4a3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


path,metrics
abfss://unity-catalog-storage@dbstorage6vxx4jf6d74oc.dfs.core.windows.net/7405609940652409/__unitystorage/catalogs/61b034ef-f999-4467-a423-10d5d7c8261f/tables/b2a4dc55-c6b2-4f25-babd-bee6e67b4d16,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1793939809894, 1793939810618, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null, 0)"


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,04ff13f2-de75-4e1e-9dae-41ac4ee34f6d,rithika_databricks_7405609940652409.default.parking_sessions_work,null,abfss://unity-catalog-storage@dbstorage6vxx4jf6d74oc.dfs.core.windows.net/7405609940652409/__unitystorage/catalogs/61b034ef-f999-4467-a423-10d5d7c8261f/tables/b2a4dc55-c6b2-4f25-babd-bee6e67b4d16,2026-11-06T04:15:20.598Z,2026-11-06T04:19:30.000Z,List(),List(),1,3135,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> b2a4dc55-c6b2-4f25-babd-bee6e67b4d16, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-56591cb8-3714-4e47-8580-f435ffe9518e, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-7261b99b-0e82-4494-bbf8-b72648dbd4a3)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


Files before: 1
Files after: 1


Section E: Structured Streaming and DLT

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS rithika_databricks_7405609940652409.default.checkpoints;

In [0]:
from pyspark.sql import functions as F

ckpt_base = "/Volumes/rithika_databricks_7405609940652409/default/checkpoints"
rate_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
display(rate_df, checkpointLocation=ckpt_base + "/q17_rate")

Checkpointing to /Volumes/rithika_databricks_7405609940652409/default/checkpoints/q17_rate


In [0]:
amount_df = (
    rate_df
    .withColumn("amount", F.col("value") * 10)
    .filter(F.col("amount") > 300)
)
display(amount_df, checkpointLocation=ckpt_base + "/q18_amount")

Checkpointing to /Volumes/rithika_databricks_7405609940652409/default/checkpoints/q18_amount


In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE parking_bronze
AS SELECT * FROM STREAM(rithika_databricks_7405609940652409.default.parking_sessions_source);

CREATE OR REFRESH STREAMING TABLE parking_silver
AS SELECT * FROM STREAM(parking_bronze)
WHERE status = 'Completed';

message
The operation was successfully executed.


In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW parking_gold
AS SELECT zone_id,
          COUNT(*) AS total_completed_sessions,
          SUM(parking_fee) AS total_revenue
FROM parking_silver
GROUP BY zone_id;

result
The operation was successfully executed.


In [0]:
%sql
SELECT * FROM rithika_databricks_7405609940652409.default.parking_gold
ORDER BY zone_id;

zone_id,total_completed_sessions,total_revenue
Z01,3,410
Z02,4,800
Z03,3,620
Z04,2,270
Z05,2,350
